# 05 — Construcción del dataset de fine-tuning

**Proyecto:** Chem RAG Assistant  
**Repositorio:** https://github.com/Jesusrodriguezf90/chem-rag-assistant  
**Fase:** Fine-tuning

---

Este notebook construye el dataset de pares en formato conversacional (`messages`)
para fine-tuning del LLM sobre el dominio de química organometálica.

In [45]:
"""
Notebook: 05_dataset.ipynb

Objetivo:
    Construir y publicar el dataset de pares prompt-completion para
    fine-tuning de un LLM especializado en química organometálica,
    a partir de los chunks y el ground truth del pipeline RAG existente.

    El dataset se construye en tres capas de calidad creciente:
      1. Pares de QA desde ground truth — alta calidad, verificados manualmente
      2. Pares de QA generados con Qwen3 sobre cada chunk — generación automática
      3. Pares de instrucción sobre conceptos clave del paper — cobertura terminológica

    Este notebook cubre:
      1. Configuración del entorno e instalación de dependencias
      2. Carga de chunks persistidos desde el pipeline RAG (02_embeddings)
      3. Construcción de pares ground truth (Capa 1 — alta calidad)
      4. Generación automática de pares QA con Qwen3 vía HF Inference API (Capa 2)
      5. Construcción de pares de instrucción sobre conceptos químicos (Capa 3)
      6. Consolidación y validación de calidad del dataset completo
      7. Persistencia local y publicación en HuggingFace Hub
      8. Resumen cuantitativo de resultados

Fuente de datos:
    Büchele WRE, Schlachta TP, Gebendorfer AL, Pamperin J, Richter LF,
    Sauer MJ, Prokop A, Kühn FE. Synthesis, characterization, and biomedical
    evaluation of ethylene-bridged tetra-NHC Pd(ii), Pt(ii) and Au(iii)
    complexes, with apoptosis-inducing properties in cisplatin-resistant
    neuroblastoma cells. Frontiers in Chemistry. 2024.
    PMC: https://pmc.ncbi.nlm.nih.gov/articles/PMC10967698/

    Documento utilizado exclusivamente con fines de investigación y desarrollo.
    No se distribuye ni se incluye en el repositorio.

Autor:   Jesús Rodríguez
Fecha:   2026-05-19
Versión: 1.0.0
"""

'\nNotebook: 05_dataset.ipynb\n\nObjetivo:\n    Construir y publicar el dataset de pares prompt-completion para\n    fine-tuning de un LLM especializado en química organometálica,\n    a partir de los chunks y el ground truth del pipeline RAG existente.\n\n    El dataset se construye en tres capas de calidad creciente:\n      1. Pares de QA desde ground truth — alta calidad, verificados manualmente\n      2. Pares de QA generados con Qwen3 sobre cada chunk — generación automática\n      3. Pares de instrucción sobre conceptos clave del paper — cobertura terminológica\n\n    Este notebook cubre:\n      1. Configuración del entorno e instalación de dependencias\n      2. Carga de chunks persistidos desde el pipeline RAG (02_embeddings)\n      3. Construcción de pares ground truth (Capa 1 — alta calidad)\n      4. Generación automática de pares QA con Qwen3 vía HF Inference API (Capa 2)\n      5. Construcción de pares de instrucción sobre conceptos químicos (Capa 3)\n      6. Consolidació

## 1. Configuración del entorno

In [46]:
# Librería estándar
import json
import os
import re
import time
from pathlib import Path

# Third-party
from datasets import Dataset
from google.colab import drive, userdata
from huggingface_hub import HfApi, InferenceClient

import sys
print(f'Python version: {sys.version}')

Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


In [47]:
!pip install datasets huggingface_hub --quiet
print('Dependencias instaladas.')

Dependencias instaladas.


In [48]:
drive.mount('/content/drive')

PROYECTO_RAIZ  = Path('/content/drive/MyDrive/chem-rag-assistant')
DIR_EMBEDDINGS = PROYECTO_RAIZ / 'data' / 'embeddings'
DIR_DATASET    = PROYECTO_RAIZ / 'data' / 'finetune'
DIR_DATASET.mkdir(parents=True, exist_ok=True)

CHUNKS_PATH = DIR_EMBEDDINGS / 'PMC10967698_chunks.json'
assert CHUNKS_PATH.exists(), f'Chunks no encontrados en {CHUNKS_PATH}'
print(f'Chunks encontrados: {CHUNKS_PATH}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Chunks encontrados: /content/drive/MyDrive/chem-rag-assistant/data/embeddings/PMC10967698_chunks.json


In [49]:
# Cargar token HF desde los secretos de Colab
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    print('HF_TOKEN cargado desde secretos de Colab.')
except Exception:
    HF_TOKEN = os.getenv('HF_TOKEN', '')
    print('HF_TOKEN cargado desde variable de entorno.')

assert HF_TOKEN, 'HF_TOKEN no encontrado. Añádelo en Colab → Secrets.'

HF_USERNAME   = 'Jesusrodriguezf90'
MODEL_DATASET = f'{HF_USERNAME}/chemistry-organometallic-qa'
print(f'Dataset destino: {MODEL_DATASET}')

HF_TOKEN cargado desde secretos de Colab.
Dataset destino: Jesusrodriguezf90/chemistry-organometallic-qa


## 2. Carga de chunks del paper

In [50]:
with open(CHUNKS_PATH, 'r', encoding='utf-8') as f:
    chunks = json.load(f)

print(f'Total chunks cargados: {len(chunks)}')
print(f'\nEjemplo chunk 0 (primeros 300 chars):')
print(chunks[0][:300])

Total chunks cargados: 77

Ejemplo chunk 0 (primeros 300 chars):
## Introduction

N-heterocyclic carbenes (NHCs),  first described in 1991, 1 have found many applications. 2 There are several structural features that allow the tuning of their electronic properties. Ring size, the adjacent heteroatoms, N -substituents, and the backbone can be modified. Changing on


## 3. Capa 1: Ground truth QA (alta calidad)

In [51]:
# Las 5 preguntas de evaluación con ground truth del proyecto RAG
# Son los pares de mayor calidad — respuestas verificadas manualmente
# Se usan también para evaluar el modelo fine-tuneado en 06b_finetune.ipynb

# SYSTEM PROMPT — define el rol del asistente para todas las capas del dataset
SYSTEM_PROMPT = (
    "You are a specialized scientific assistant in organometallic chemistry "
    "and medicinal inorganic chemistry. Answer questions accurately and "
    "concisely based on your knowledge of NHC metal complexes."
)

GROUND_TRUTH_QA = [
    {
        "question": "What metals are used in the NHC complexes studied?",
        "answer": (
            "The NHC complexes studied contain palladium (Pd), platinum (Pt) "
            "and gold (Au) as metal centers. Specifically, Pd(II), Pt(II) and "
            "Au(III) tetracarbene complexes were synthesized using ethylene-bridged "
            "tetradentate NHC ligands."
        ),
        "fuente": "ground_truth",
        "calidad": "alta"
    },
    {
        "question": (
            "What is the effect of the complexes on cisplatin-resistant "
            "neuroblastoma cells?"
        ),
        "answer": (
            "AuL9 induces apoptosis in cisplatin-resistant SK-N-AS neuroblastoma "
            "cells in vitro via the mitochondrial and ROS pathway. The complex "
            "overcomes cisplatin resistance, suggesting that procaspase-8 plays "
            "a minor role in AuL9-induced apoptosis."
        ),
        "fuente": "ground_truth",
        "calidad": "alta"
    },
    {
        "question": "What analytical techniques were used to characterize the compounds?",
        "answer": (
            "The compounds were characterized by NMR spectroscopy (1H, 13C, 19F), "
            "elemental analysis (C/H/N) at the Microanalytical Laboratory of TUM, "
            "and electrospray ionization mass spectrometry (ESI-MS and HR-ESI-MS) "
            "on a Thermo Fisher Orbitrap. Single-crystal X-ray diffraction (SC-XRD) "
            "was used for structural characterization of PdL3, PtL3 and PdL9."
        ),
        "fuente": "ground_truth",
        "calidad": "alta"
    },
    {
        "question": "What is the role of the ethylene bridge in the tetra-NHC ligand design?",
        "answer": (
            "The ethylene bridge connects the NHC units to form cyclic tetradentate "
            "ligands. It introduces a +I inductive effect that increases electron "
            "density on the carbene carbon, leading to upfield shifts in NMR. The "
            "bridge geometry forces the ligand into a macrocyclic structure that "
            "enables tetracarbene coordination to a single metal center."
        ),
        "fuente": "ground_truth",
        "calidad": "alta"
    },
    {
        "question": "How do the cytotoxicity results of the Au(III) complexes compare to cisplatin?",
        "answer": (
            "AuL9 shows higher cytotoxicity than cisplatin in cisplatin-resistant "
            "SK-N-AS neuroblastoma cells. While cisplatin fails in resistant cells, "
            "AuL9 induces significant apoptosis and inhibits proliferation in a "
            "dose-dependent manner, with nearly 100% inhibition at 50 μM."
        ),
        "fuente": "ground_truth",
        "calidad": "alta"
    },
]

# Formatear como mensajes conversacionales
# Se usa el formato messages en lugar de prompt/completion string por dos razones:
#   1. SFTTrainer aplica automáticamente el chat template correcto del modelo
#      en el momento del entrenamiento, evitando errores de tokenización
#   2. Es compatible con cualquier modelo (Qwen, TinyLlama, Mistral)
#      sin cambiar el dataset
# Los tokens para Qwen son <|im_start|>/<|im_end|>
pares_ground_truth = []
for qa in GROUND_TRUTH_QA:
    par = {
        "messages": [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": qa["question"]},
            {"role": "assistant", "content": qa["answer"]},
        ],
        "fuente":  qa["fuente"],
        "calidad": qa["calidad"],
    }
    pares_ground_truth.append(par)

print(f'Pares ground truth generados: {len(pares_ground_truth)}')
print(f'\nEjemplo par 0:')
print(f'USER:      {pares_ground_truth[0]["messages"][1]["content"][:80]}')
print(f'ASSISTANT: {pares_ground_truth[0]["messages"][2]["content"][:100]}...')

Pares ground truth generados: 5

Ejemplo par 0:
USER:      What metals are used in the NHC complexes studied?
ASSISTANT: The NHC complexes studied contain palladium (Pd), platinum (Pt) and gold (Au) as metal centers. Spec...


## 4. Capa 2: QA generado desde chunks con Qwen3

In [52]:
cliente = InferenceClient(
    provider="auto",
    api_key=HF_TOKEN,
)

def generar_qa_desde_chunk(chunk: str) -> dict | None:
    """Genera un par QA a partir de un chunk usando Qwen3 vía InferenceClient.

    Usa InferenceClient de huggingface_hub que gestiona automáticamente
    el routing al Inference Provider correcto y aplica el chat template
    de Qwen3.

    Temperatura 0.7 — valor mínimo recomendado por Qwen3 en modo non-thinking
    (documentación oficial: temperature=0.7, top_p=0.8, top_k=20).
    Valores inferiores causan bucles de repetición con este modelo.
    La precisión factual se garantiza mediante el prompt que exige citar
    detalles específicos del texto y el filtro de longitud mínima.

    Args:
        chunk: texto del chunk del paper.

    Returns:
        Diccionario con question y answer, o None si la generación falla.
    """
    try:
        respuesta = cliente.chat.completions.create(
            model="Qwen/Qwen3-8B",
            messages=[
                {
                    "role": "user",
                    "content": (
                        f"Given the following excerpt from a scientific paper on "
                        f"organometallic chemistry, generate ONE specific question "
                        f"and its detailed answer.\n\n"
                        f"Rules:\n"
                        f"- The question must be answerable ONLY from the provided text\n"
                        f"- The answer must be factual and cite specific details\n"
                        f"- Format your response as:\n"
                        f"QUESTION: <your question>\n"
                        f"ANSWER: <your answer>\n\n"
                        f"Text excerpt:\n{chunk[:600]}\n\n"
                        f"/no_think"
                    )
                }
            ],
            max_tokens=250,
            temperature=0.7,
            top_p=0.8,
        )

        texto = respuesta.choices[0].message.content

        # Eliminar bloque <think> si /no_think no fue respetado
        texto = re.sub(r'<think>.*?</think>\s*', '', texto, flags=re.DOTALL).strip()

        if 'QUESTION:' in texto and 'ANSWER:' in texto:
            partes          = texto.split('ANSWER:')
            pregunta        = partes[0].replace('QUESTION:', '').strip()
            respuesta_texto = partes[1].strip() if len(partes) > 1 else ''

            if len(pregunta) > 20 and len(respuesta_texto) > 50:
                return {"question": pregunta, "answer": respuesta_texto}

        return None

    except Exception as e:
        print(f'ERROR: {e}')
        return None

# Cargar pares ya generados si existen — evita consumir créditos innecesariamente
RUTA_CHUNK_QA = DIR_DATASET / 'chunk_qa_parcial.json'

if RUTA_CHUNK_QA.exists() and RUTA_CHUNK_QA.stat().st_size > 10:
    with open(RUTA_CHUNK_QA, 'r', encoding='utf-8') as f:
        pares_chunk_qa = json.load(f)
    chunks_ya_procesados = len(pares_chunk_qa)
    print(f'Cargados {chunks_ya_procesados} pares desde disco — reanudando desde chunk {chunks_ya_procesados}')
else:
    pares_chunk_qa       = []
    chunks_ya_procesados = 0
    print('Sin pares previos — iniciando desde el principio')

chunks_procesados = chunks_ya_procesados
chunks_fallidos   = 0

# Si ya tenemos todos los pares necesarios, no ejecutar el bucle
if chunks_ya_procesados >= 60:
    print(f'Dataset completo — {chunks_ya_procesados} pares ya generados.')
    print('Elimina el archivo chunk_qa_parcial.json para regenerar.')
else:
    # Reanudar desde el chunk donde se quedó — no reprocesa los ya guardados
    # ni consume créditos si el dataset ya está completo
    for i, chunk in enumerate(chunks[:60]):
        if i < chunks_ya_procesados:
            continue

        print(f'[{i+1:03d}/{min(60, len(chunks))}] Generando QA...', end=' ')

        resultado = generar_qa_desde_chunk(chunk)

        if resultado:
            par = {
                "messages": [
                    {"role": "system",    "content": SYSTEM_PROMPT},
                    {"role": "user",      "content": resultado["question"]},
                    {"role": "assistant", "content": resultado["answer"]},
                ],
                "fuente":  "chunk_qa",
                "calidad": "media",
            }
            pares_chunk_qa.append(par)

            # Guardar en disco inmediatamente tras cada par exitoso
            # Garantiza que no se pierden pares si los créditos se agotan
            with open(RUTA_CHUNK_QA, 'w', encoding='utf-8') as f:
                json.dump(pares_chunk_qa, f, ensure_ascii=False, indent=2)

            print(f'OK — {resultado["question"][:60]}...')
            chunks_procesados += 1
        else:
            print('FALLIDO')
            chunks_fallidos += 1

        time.sleep(0.5)

print(f'\nPares chunk QA generados : {chunks_procesados}')
print(f'Chunks fallidos           : {chunks_fallidos}')
print(f'Guardados en             : {RUTA_CHUNK_QA}')

Cargados 60 pares desde disco — reanudando desde chunk 60
Dataset completo — 60 pares ya generados.
Elimina el archivo chunk_qa_parcial.json para regenerar.

Pares chunk QA generados : 60
Chunks fallidos           : 0
Guardados en             : /content/drive/MyDrive/chem-rag-assistant/data/finetune/chunk_qa_parcial.json


## 5. Capa 3: Pares de instrucción sobre conceptos químicos

In [53]:
# Pares de instrucción sobre conceptos clave del paper
# Cubren terminología y conceptos que los chunks pueden no abordar directamente
# Escritos manualmente para garantizar precisión química

CONCEPT_QA = [
    {
        "question": "What are N-heterocyclic carbenes (NHCs) and why are they important in coordination chemistry?",
        "answer": (
            "N-heterocyclic carbenes (NHCs) are stable carbene ligands containing "
            "a divalent carbon atom within a nitrogen-containing heterocyclic ring. "
            "First described in 1991, they are strong sigma-donors that form stable "
            "metal-carbon bonds. Their electronic properties can be tuned by modifying "
            "the ring size, N-substituents, adjacent heteroatoms, and backbone saturation, "
            "making them versatile ligands for catalysis and medicinal chemistry."
        ),
    },
    {
        "question": "What is the difference between imidazole and imidazoline NHC backbones?",
        "answer": (
            "Imidazole-based NHCs have an unsaturated backbone with partial aromaticity, "
            "which increases NHC stability by approximately 100 kJ/mol. Imidazoline-based "
            "NHCs have a saturated backbone that concentrates electron density on the C2 "
            "carbene carbon due to the absence of pi-interactions, theoretically making "
            "them stronger sigma-donors. This difference affects NMR chemical shifts "
            "and reactivity of the resulting metal complexes."
        ),
    },
    {
        "question": "What is the transmetalation route using silver oxide in NHC complex synthesis?",
        "answer": (
            "The silver transmetalation route involves first forming a silver-NHC complex "
            "in situ using Ag2O as a mild base to deprotonate the imidazolium salt. The "
            "resulting Ag-NHC species then undergoes transmetalation with the target metal "
            "precursor (e.g., Pd(OAc)2, KAuCl4) to give the desired complex. This route "
            "is used when direct metalation fails, as silver acts as a carbene transfer agent."
        ),
    },
    {
        "question": "What does ORTEP representation show in crystallographic studies?",
        "answer": (
            "ORTEP (Oak Ridge Thermal Ellipsoid Plot) representations show the three-dimensional "
            "structure of a molecule from single-crystal X-ray diffraction data. Thermal "
            "ellipsoids represent the probability distribution of atomic positions at a "
            "given probability level (typically 50%). Bond lengths and angles extracted "
            "from ORTEP structures provide precise geometric parameters for metal complexes."
        ),
    },
    {
        "question": "What is the significance of the square planar geometry in Pd(II) NHC complexes?",
        "answer": (
            "Pd(II) is a d8 metal that preferentially adopts square planar coordination "
            "geometry. In tetracarbene complexes, the four NHC ligands coordinate to the "
            "Pd center in a square planar arrangement. The Pd-C carbene bond lengths "
            "(approximately 2.0-2.1 Angstrom) and the C-Pd-C angles (close to 90 and 180 "
            "degrees) confirm the square planar geometry, which is related to the "
            "biological activity of these complexes."
        ),
    },
    {
        "question": "What is apoptosis and how is it different from necrosis in the context of cancer treatment?",
        "answer": (
            "Apoptosis is programmed cell death characterized by nuclear DNA fragmentation, "
            "cell shrinkage, and membrane blebbing without inflammation. Necrosis is "
            "uncontrolled cell death that releases intracellular contents and causes "
            "inflammation. In cancer treatment, apoptosis induction is preferred because "
            "it selectively eliminates cancer cells without damaging surrounding tissue. "
            "LDH release assays distinguish necrosis from apoptosis by detecting membrane "
            "integrity loss."
        ),
    },
    {
        "question": "What role do reactive oxygen species (ROS) play in AuL9-induced apoptosis?",
        "answer": (
            "ROS play a critical role in AuL9-induced apoptosis in SK-N-AS neuroblastoma "
            "cells. N-acetylcysteine (NAC), a known ROS inhibitor, significantly reduces "
            "AuL9-induced apoptosis, confirming ROS involvement. The mitochondrial pathway "
            "is also implicated, as AuL9 impairs mitochondrial membrane potential. However, "
            "the exact mechanism by which AuL9 generates or triggers ROS production "
            "requires further investigation."
        ),
    },
    {
        "question": "What is the ethylene bistriflate reagent and what role does it play in macrocycle synthesis?",
        "answer": (
            "Ethylene bistriflate (ethylene bis(trifluoromethanesulfonate)) is a bifunctional "
            "alkylating agent used to bridge two imidazole or imidazoline units via their "
            "nitrogen atoms. The reaction is performed at -45 degrees Celsius under dry "
            "conditions to favor formation of the 20-membered macrocyclic C[4] product "
            "over the 30-membered C[6] oligomer. Slow addition over 5-6 hours at low "
            "temperature is critical to achieve high C[4] selectivity (up to 98-100%)."
        ),
    },
]

# Formatear en formato messages — coherente con el resto del dataset
# SFTTrainer aplica automáticamente el chat template correcto del modelo
pares_concept = []
for qa in CONCEPT_QA:
    par = {
        "messages": [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": qa["question"]},
            {"role": "assistant", "content": qa["answer"]},
        ],
        "fuente":  "concept",
        "calidad": "alta",
    }
    pares_concept.append(par)

print(f'Pares de concepto generados: {len(pares_concept)}')

Pares de concepto generados: 8


## 6. Consolidación y validación del dataset

In [54]:
# Consolidar las tres capas en un único dataset
todos_pares = pares_ground_truth + pares_chunk_qa + pares_concept

# Validación de calidad mínima sobre el formato messages
# Comprueba longitud de pregunta y respuesta, presencia de interrogación
# y que la respuesta no sea excesivamente larga (posible alucinación)
pares_validos = [
    p for p in todos_pares
    if len(p['messages'][1]['content']) > 20        # pregunta no trivial
    and len(p['messages'][2]['content']) > 50       # respuesta suficientemente detallada
    and '?' in p['messages'][1]['content']          # formato de pregunta correcto
    and len(p['messages'][2]['content']) < 2000     # respuesta no excesivamente larga
]

print('=' * 50)
print('RESUMEN DEL DATASET')
print('=' * 50)
print(f'Capa 1 — Ground truth  : {len(pares_ground_truth)}')
print(f'Capa 2 — Chunk QA      : {len(pares_chunk_qa)}')
print(f'Capa 3 — Conceptos     : {len(pares_concept)}')
print(f'Total pares            : {len(todos_pares)}')
print(f'Pares válidos          : {len(pares_validos)}')
print(f'Pares descartados      : {len(todos_pares) - len(pares_validos)}')
print(f'Calidad alta           : {sum(1 for p in pares_validos if p["calidad"] == "alta")}')
print(f'Calidad media          : {sum(1 for p in pares_validos if p["calidad"] == "media")}')
print('=' * 50)

RESUMEN DEL DATASET
Capa 1 — Ground truth  : 5
Capa 2 — Chunk QA      : 60
Capa 3 — Conceptos     : 8
Total pares            : 73
Pares válidos          : 73
Pares descartados      : 0
Calidad alta           : 13
Calidad media          : 60


## 7. Persistencia local y publicación en HF Hub

In [55]:
RUTA_DATASET = DIR_DATASET / 'chemistry_qa_dataset.json'

if RUTA_DATASET.exists():
    print(f'Dataset ya existe localmente — omitiendo guardado y publicación.')
    print(f'Elimina {RUTA_DATASET.name} para regenerar.')
else:
    # Guardar dataset localmente
    with open(RUTA_DATASET, 'w', encoding='utf-8') as f:
        json.dump(pares_validos, f, ensure_ascii=False, indent=2)

    print(f'Dataset guardado localmente: {RUTA_DATASET}')
    print(f'Tamaño: {RUTA_DATASET.stat().st_size / 1024:.1f} KB')

    # Publicar en HuggingFace Hub — solo si es la primera vez
    hf_dataset = Dataset.from_list(pares_validos)
    hf_dataset.push_to_hub(
        repo_id=MODEL_DATASET,
        token=HF_TOKEN,
        private=False,
    )
    print(f'\nDataset publicado en HF Hub: https://huggingface.co/datasets/{MODEL_DATASET}')

Dataset ya existe localmente — omitiendo guardado y publicación.
Elimina chemistry_qa_dataset.json para regenerar.


## 8. Resumen final

In [56]:
print('=' * 60)
print('RESUMEN — DATASET DE FINE-TUNING COMPLETADO')
print('=' * 60)
print(f'  Documento origen      : PMC10967698')
print(f'  Total pares válidos   : {len(pares_validos)}')
print(f'  Calidad alta          : {sum(1 for p in pares_validos if p["calidad"] == "alta")}')
print(f'  Calidad media         : {sum(1 for p in pares_validos if p["calidad"] == "media")}')
print(f'  Dataset HF Hub        : {MODEL_DATASET}')
print(f'  Dataset local         : {RUTA_DATASET}')
print('=' * 60)
print('Siguiente paso: 06b_finetune.ipynb — QLoRA fine-tuning')
print('=' * 60)

RESUMEN — DATASET DE FINE-TUNING COMPLETADO
  Documento origen      : PMC10967698
  Total pares válidos   : 73
  Calidad alta          : 13
  Calidad media         : 60
  Dataset HF Hub        : Jesusrodriguezf90/chemistry-organometallic-qa
  Dataset local         : /content/drive/MyDrive/chem-rag-assistant/data/finetune/chemistry_qa_dataset.json
Siguiente paso: 06b_finetune.ipynb — QLoRA fine-tuning
